In [ ]:
import numpy as np
import cv2

camera_matrix = np.load("camera_matrix.npy")
dist_coeffs = np.load("dist_coeffs.npy")
R_shelf = np.load("R_shelf.npy")
T_shelf = np.load("T_shelf.npy")

def pixel_to_shelf(cx, cy, shelf_z=0):
    """
    Convert YOLO pixel center (cx, cy) into real-world coordinates (X, Y, Z)
    on the Z = shelf_z plane.

    Returns plain Python floats.
    """

    # --- 1. Undistort pixel → normalized camera coordinates ---
    undistorted = cv2.undistortPoints(
        np.array([[[cx, cy]]], dtype=np.float32),
        camera_matrix,
        dist_coeffs
    )[0][0]

    Xn, Yn = float(undistorted[0]), float(undistorted[1])
    ray_cam = np.array([Xn, Yn, 1.0], dtype=float)  # homogeneous camera ray

    # --- 2. Compute the camera origin in shelf coordinates ---
    C_shelf = -(R_shelf.T @ T_shelf).reshape(3)

    # --- 3. Convert ray into shelf coordinates ---
    ray_shelf = (R_shelf.T @ ray_cam).reshape(3)

    # Prevent divide-by-zero if ray parallel to plane
    if abs(ray_shelf[2]) < 1e-6:
        raise ZeroDivisionError("Camera ray is parallel to the shelf plane (Z).")

    # --- 4. Intersection of camera ray with plane Z = shelf_z ---
    t = (shelf_z - C_shelf[2]) / ray_shelf[2]

    P = C_shelf + t * ray_shelf  # (X, Y, Z)

    # Always return floats (not numpy arrays)
    return float(P[0]), float(P[1]), float(P[2])

